# 유튜브 쇼츠 댓글 수집하기

In [ ]:
#필요한 모듈 불러오기
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time, math, os, urllib, urllib.request, re
import pandas  as pd

In [2]:
# 키워드 검색받고 저장할 폴더와 파일명 설정
print('='*100)
print('유튜브 영상 댓글정보 수집')
print('='*100)
print('\n')

query_txt = '유튜브 쇼츠 댓글'
query_url = input('1. 댓글을 크롤링할 유튜브쇼츠의 전체 url을 입력하세요:')
if query_url == '' :
    query_url = 'https://www.youtube.com/shorts/UkU_P-jfBtM'

try:
    cnt = int(input('2. 크롤링 할 건수는 몇건 입니까? (0 입력시, 전체 건수 크롤링 / 기본값: 5):'))
except:
    cnt = 5

f_dir = input('3. 파일을 저장할 폴더명만 작성하세요. (예: C:\\Users\\ITWILL\\workspace\\seo\\크롤링\\)')
if f_dir == '' :
    f_dir = 'C:\\Users\\ITWILL\\workspace\\seo\\크롤링\\'

# 저장될 파일 위치와 이름 지정
n = time.localtime()
s = f'{n.tm_year}-{n.tm_mon}-{n.tm_mday}-{n.tm_hour}-{n.tm_min}-{n.tm_sec}'

os.makedirs(f_dir+s+'-'+query_txt)
os.chdir(f_dir+s+'-'+query_txt)

ff_name = f_dir + s + '-' + query_txt + '\\' + s + '-' + query_txt + '.txt'
fc_name = f_dir + s + '-' + query_txt + '\\' + s + '-' + query_txt + '.csv'
fx_name = f_dir + s + '-' + query_txt + '\\' + s + '-' + query_txt + '.xlsx'

유튜브 영상 댓글정보 수집




In [3]:
# 크롬 드라이버를 사용하여 웹 브라우저 실행
s_time = time.time()

s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

driver.get(query_url)
driver.maximize_window()
time.sleep(3)

In [4]:
# 현재 총 댓글 건수를 확인하여 사용자의 요청 건수와 비교 후 동기화
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

#str -> int로 변경
result = soup.find_all('label', 'yt-spec-button-shape-with-label')[2]
result2 = result.get_text()

print('=' * 80)
result3 = result2.replace(',','')
result4 = re.search('\d+', result3)
search_cnt = int(result4.group())

#cnt = 0 일때, 전체 조회 조건
if cnt > search_cnt:
    cnt = search_cnt
elif cnt == 0:
    cnt = search_cnt

page_cnt = math.ceil(cnt / 15)

print('전체 검색 결과 건수 : ', search_cnt,'건')
print('실제 최종 출력 건수', cnt)
print('실제 출력될 최종 페이지 수', page_cnt)


전체 검색 결과 건수 :  22 건
실제 최종 출력 건수 22
실제 출력될 최종 페이지 수 2


<>:11: SyntaxWarning: invalid escape sequence '\d'
<>:11: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ITWILL\AppData\Local\Temp\ipykernel_8568\688173250.py:11: SyntaxWarning: invalid escape sequence '\d'
  result4 = re.search('\d+', result3)


In [5]:
# 댓글창 열기
driver.find_element(By.XPATH, '//*[@id="button-bar"]/reel-action-bar-view-model/button-view-model[1]/label/button/yt-touch-feedback-shape').click()
#driver.find_element(By.XPATH,'//*[@id="cbox_module"]/div/div[9]/a/span[1]').click()
time.sleep(1)

In [6]:
c = driver.find_element(By.XPATH, '//div[@id="contents" and contains(@class, " style-scope ytd-item-section-renderer style-scope ytd-item-section-renderer")]')

def scroll_down(driver, c=None):
    driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight;", c)
    time.sleep(2.5)

for a in range(1, page_cnt + 1):

    if a == page_cnt:
        break
    else :
        scroll_down(driver, c)

        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')


id2 = []                # 작성자명 (닉네임)
date2 = []              # 댓글 작성일자
ny2 = []                # 댓글 내용
good2 = []             # 좋아요 횟수

count = 0

print('이제 리뷰 정보를 수집합니다.')

html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

reple_result = soup.select('ytd-comment-thread-renderer')

for li in reple_result:
    
    count += 1
    
    print(f'총 {cnt}건 중 {count}번째 댓글 수집 중입니다.')

    f = open(ff_name, 'a', encoding='utf-8')
    
    print('\n')

    id1 = li.find(id='author-text').get_text().replace('\n','')
    print('1. 작성자ID : ', id1)
    f.write('\n')
    f.write(f'총 {cnt}건 중 {count}번째 댓글 수집 중입니다.' + '\n')
    f.write('1. 작성자ID : ' + id1 + '\n')
    id2.append(id1)

    try:
        date = li.find(id = 'published-time-text').get_text().replace('\n','').strip()
    except Exception as e:
        date = '작성자에 의해 삭제된 댓글입니다.'
        print('2. 작성일자 : ', date)
    else : 
        print('2. 작성일자 : ', date)
    f.write('2. 작성일자 : '+ date + '\n')
    date2.append(date)

    ny = li.find(id = 'content-text').get_text()
    print('3. 댓글 : ', ny)
    f.write('3. 댓글 : '+ ny + '\n')
    ny2.append(ny)
  
    try : 
        good = li.find(id = 'vote-count-middle').get_text().replace('\n','').strip()
    except Exception as e:
        if good == '':
            good = '0'
        print('4. 좋아요 수 : ', good)
    else : 
        print('4. 좋아요 수 : ', good)
    f.write('4. 좋아요 수 : '+ good + '\n')
    good2.append(good)

    f.close()

    if count == cnt:
        break

print(f'''총 {cnt}건 중, 댓글은 {count}건, 대댓글은{cnt-count}건 으로 확인됩니다.
      수집된 총 댓글은 {count}건 입니다.''')

이제 리뷰 정보를 수집합니다.
총 22건 중 1번째 댓글 수집 중입니다.


1. 작성자ID :   @catch_speed 
2. 작성일자 :  6개월 전
3. 댓글 :  올리브영 알바생이 추천하는 꿀템 5가지

1. 짱구 향수
짱구덕후라면 꼭 샤야된다. 짱구도 귀엽고 파우치 키링까지 주는데 포근한 코튼향이라 내 살냄새같아서 짱조음

2. 콜레올로지 컷팅 젤리 

아이돌 다이어트 필수템인 컷팅 젤리인데 간식 땡길 때 쭉쭉 짜먹으면 좋아요! 체지방 컷팅에 식후혈당까지 다잡아줘서 가방에 하나씩 넣고다니기 좋음! 

3. 도로시와 퍼스널 컬러 컷 누드브라 
너무너무 좋아서 여행갈때도 챙겨갔는데 퍼스널 컬러 컷 누드브라 이름처럼 컬러가 여러개라 
내 피부톤에 찰떡인 컬러로 고르면 티 1도 안남!! 촉감도 부드럽고 얇아서 역대급으로 자연스럽고 라인도이쁨♡

4. 메디큐브 핑크콜라겐 캡슐크림
저 이제 피부과 안갑니다. 분홍색 캡슐을 톡톡 터트리며 발라주면 담날 트러블 다들어가요

5. 레인보우 치즈 베이글칩
치즈맛 제대로나고 바삭바삭한게 진짜 맛있어서 입심심할때 다이어트 과자로 좋아요
4. 좋아요 수 :  32
총 22건 중 2번째 댓글 수집 중입니다.


1. 작성자ID :   @딩동-d3q1 
2. 작성일자 :  9일 전
3. 댓글 :  저거 캡슐크림 달바꺼도 진짜 좋아요!! 비타민이 진짜 최고.. 갠적으로 스킨케어 할 때 비타민 성분만 쓰는데 바이오힐보 비타민토너, 비타치올 비타민앰플, 달바 비타크림으로 관리하고나서 트러블 자국, 잡티 이런거 싹 옅어지고 피부 톤 진짜 맑아짐 + 메가도스로 고려은단 비타민c도 2천 미리 하루 3번 총 6천 복용중 입니다!!
4. 좋아요 수 :  49
총 22건 중 3번째 댓글 수집 중입니다.


1. 작성자ID :   @정휴닝 
2. 작성일자 :  6개월 전
3. 댓글 :  저 짱구 향수.. 그냥 목욕탕 냄새나요;;
진짜 ㄹㅇ 찐 목욕탕 냄새... 뭔 샤워하고 갓나온 향이라는지
4. 좋아요 수 :  123
총 22건 중 4번째 댓글

In [7]:
#csv , xlsx 형태로 저장하기              
shorts = pd.DataFrame()
shorts['작성자ID']=id2
shorts['작성일자']=pd.Series(date2)
shorts['댓글']=pd.Series(ny2)
shorts['좋아요 수']=pd.Series(good2)

# csv 형태로 저장하기
shorts.to_csv(fc_name,encoding="utf-8-sig",index=False)

# 엑셀 형태로 저장하기
shorts.to_excel(fx_name ,index=False)

e_time = time.time( )
t_time = e_time - s_time

count -= 1
print("\n")
print("=" *100)
print("1.요청된 총 %s 건의 리뷰 중에서 실제 크롤링 된 리뷰 수는 %s 건입니다" %(cnt,count))
print("2.총 소요시간은 %s 초 입니다 " %round(t_time,1))
print("3.파일 저장 완료: txt 파일명 : %s " %ff_name)
print("4.파일 저장 완료: csv 파일명 : %s " %fc_name)
print("5.파일 저장 완료: xlsx 파일명 : %s " %fx_name)
print("=" *100)




1.요청된 총 22 건의 리뷰 중에서 실제 크롤링 된 리뷰 수는 14 건입니다
2.총 소요시간은 10.5 초 입니다 
3.파일 저장 완료: txt 파일명 : C:\Users\ITWILL\workspace\seo\크롤링\2026-3-23-17-36-21-유튜브 쇼츠 댓글\2026-3-23-17-36-21-유튜브 쇼츠 댓글.txt 
4.파일 저장 완료: csv 파일명 : C:\Users\ITWILL\workspace\seo\크롤링\2026-3-23-17-36-21-유튜브 쇼츠 댓글\2026-3-23-17-36-21-유튜브 쇼츠 댓글.csv 
5.파일 저장 완료: xlsx 파일명 : C:\Users\ITWILL\workspace\seo\크롤링\2026-3-23-17-36-21-유튜브 쇼츠 댓글\2026-3-23-17-36-21-유튜브 쇼츠 댓글.xlsx 


In [8]:
#DB연동을 위한, 코드
import excelrd, pymysql

xlsx = excelrd.open_workbook(fx_name)
sheet = xlsx.sheet_by_name('Sheet1')

mysql = pymysql.connect(host='192.168.25.17', port=3306,
                        user='my_user',
                        passwd='mysql',
                        db='my_testdb',
                        charset='utf8')

cursor = mysql.cursor()

query_drop = "DROP TABLE IF EXISTS y_shorts"
cursor.execute(query_drop)

#테이블 생성
query_1 = '''
CREATE TABLE y_shorts(
u_id TEXT,
date TEXT,
ny TEXT,
good TEXT
);
'''
cursor.execute(query_1)

# DB 삽입
query_2 = '''INSERT INTO y_shorts(
u_id, date, ny, good) VALUES(%s, %s, %s, %s)'''

for r in range(1, sheet.nrows):
    u_id    = sheet.cell(r, 0).value
    date = sheet.cell(r, 1).value
    ny = sheet.cell(r, 2).value
    good = sheet.cell(r, 3).value  

    values = (u_id, date, ny, good)
    cursor.execute(query_2, values)

mysql.commit()  # 저장
cursor.close()  # 닫기
mysql.close()   # 닫기